# D05 · I3C - what changed from I2C and why

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D05_i3c_what_changed_from_i2c_and_why.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 45 min

**Skill Hub concepts:** I3C (Improved Inter-Integrated Circuit); I3C dynamic address assignment; I3C in-band interrupt and hot-join; I3C common command codes; I3C high data rate modes; I3C versus I2C trade-offs

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D05")

## Map: I2C vs I3C, feature by feature

| Feature | I2C | I3C (MIPI) |
|---|---|---|
| Data drive | open-drain both ways | open-drain for arbitration/addressing, then **push-pull** |
| Speed | 0.1 / 0.4 / 1 MHz | SDR up to 12.5 MHz; HDR modes (DDR, ternary TSP/TSL, bulk BT) multiply throughput |
| Addresses | fixed 7-bit, set by pins | **dynamic**: controller assigns them (ENTDAA, or SETDASA from a static address) |
| Interrupts | extra GPIO per device | **in-band interrupt (IBI)**: target starts a header on SDA, lowest address wins |
| Late devices | not supported | **hot-join**: target requests the reserved address 0x02, then gets one via DAA |
| Management | vendor registers | **common command codes (CCC)** on broadcast address 0x7E |
| Legacy | | I2C targets may share the bus if they have the 50 ns spike filter |

Dynamic address assignment (ENTDAA): every unassigned target shifts out 64 bits, MSB first, open-drain:
**48-bit provisioned ID | BCR | DCR**. A 0 beats a 1 (wired-AND), so the lowest 64-bit value wins, gets the next
address, drops out, and the process repeats until nobody acknowledges.

| CCC | Code | Type |
|---|---|---|
| ENEC / DISEC (events on/off) | 0x00 / 0x01 | broadcast |
| RSTDAA (forget dynamic addresses) | 0x06 | broadcast |
| ENTDAA (start address assignment) | 0x07 | broadcast |
| ENTHDR0 (enter HDR-DDR) | 0x20 | broadcast |
| ENEC / DISEC | 0x80 / 0x81 | direct |
| SETDASA (static to dynamic) | 0x87 | direct |
| GETPID / GETBCR / GETDCR | 0x8D / 0x8E / 0x8F | direct |

Rule: codes 0x00 to 0x7F are broadcast, 0x80 to 0xFE are direct.

## Task 1: run dynamic address assignment
`targets` is a list of `(pid48, bcr, dcr)`. Simulate the bit-by-bit open-drain arbitration (do not just call
`min`) and return `{pid48: dynamic_address}`, handing out addresses from `first` upward.

In [ ]:
def entdaa(targets, first=0x08):
    raise NotImplementedError  # TODO

lab.check("daa", entdaa)

In [ ]:
%%writefile d05_daa.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d05.h
// Grade it with lab.check("daa", "d05_daa.c"). For C++, name the file .cpp in both places.
#include "d05.h"

void entdaa(const I3cTarget *targets, size_t n, uint8_t first, uint8_t *addr_out) {
    (void)targets; (void)first;
    for (size_t i = 0; i < n; ++i) addr_out[i] = 0; /* TODO: bitwise wired-AND arbitration on PID|BCR|DCR */
}

In [ ]:
# lab.check("daa", "d05_daa.c")  # uncomment after writing the C/C++ cell above

## Task 2: classify CCCs and settle simultaneous in-band interrupts

In [ ]:
def ccc_type(code):
    raise NotImplementedError  # TODO

def ibi_winner(dynamic_addresses):
    """Several targets start an IBI at once; the address header arbitrates like DAA. Return the winner."""
    raise NotImplementedError  # TODO

lab.check("ccc_ibi", ccc_type, ibi_winner)

In [ ]:
%%writefile d05_ccc_ibi.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d05.h
// Grade it with lab.check("ccc_ibi", "d05_ccc_ibi.c"). For C++, name the file .cpp in both places.
#include "d05.h"

bool ccc_is_direct(uint8_t code) { (void)code; return false; /* TODO */ }

uint8_t ibi_winner(const uint8_t *dynamic_addresses, size_t n) {
    (void)dynamic_addresses; (void)n;
    return 0; /* TODO */
}

In [ ]:
# lab.check("ccc_ibi", "d05_ccc_ibi.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Remember).** Maximum SDR clock in MHz?

**M2 (Apply).** Bits each target shifts out during one DAA arbitration round?

**M3 (Apply).** Targets with dynamic addresses 0x0A, 0x09 and 0x0C raise an IBI at the same instant. Which address wins?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** What does hot-join allow?
- a) two controllers to share a bus
- b) a target powered up after bus initialization to request a dynamic address without a bus reset
- c) joining two I3C buses with a bridge

**K2 (Evaluate).** You are designing the sensor bus for a robot arm head (IMU, magnetometer, two temperature sensors,
one legacy I2C EEPROM). Argue for or against I3C: wiring, interrupts, speed, legacy parts and firmware effort.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d05.h`).
3. In `roboprep/checks/d05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()